In [0]:
passengers_day1 = [
(101,"Rahul Sharma","Hyderabad","Economy","India"),
(102,"Priya Reddy","Bangalore","Business","India"),
(103,"Amit Kumar","Mumbai","Economy","India"),
(104,"Sneha Patel","Delhi","Premium Economy","India"),
(105,"Farhan Ali","Chennai","Economy","India")
]
columns = [
"passenger_id",
"passenger_name",
"city",
"travel_class",
"country"
]
df_day1 = spark.createDataFrame(
passengers_day1,
columns
)

In [0]:
df_day1.write.format("delta") \
    .mode("overwrite") \
    .save("/tmp/passengers_delta")

In [0]:
df_day1.write.format("delta") \
    .mode("overwrite") \
    .save("/tmp/passengers_delta")

In [0]:
spark.read.format("delta") \
    .load("/tmp/passengers_delta") \
    .count()

5

In [0]:
df = spark.read.format("delta") \
    .load("/tmp/passengers_delta")

display(df)

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
102,Priya Reddy,Bangalore,Business,India
103,Amit Kumar,Mumbai,Economy,India
104,Sneha Patel,Delhi,Premium Economy,India
105,Farhan Ali,Chennai,Economy,India


In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(
    spark,
    "/tmp/passengers_delta"
)

delta_table.history().show(truncate=False)

+-------+-------------------+---------------+-------------------------------------------------------+---------+------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------+------------+------------------------------------------+
|version|timestamp          |userId         |userName                                               |operation|operationParameters                                         |job |notebook          |queryHistoryStatementId             |clusterId               |readVersion|isolationLevel   |isBlindAppend|operationMetrics                                                                                                                          |userMetadata|engineInfo                                |
+---

In [0]:
passengers_day2 = [
(102,"Priya Reddy","Bangalore","First Class","India"),
(104,"Sneha Patel","Hyderabad","Premium Economy","India"),
(106,"Neha Singh","Pune","Economy","India"),
(107,"Arjun Verma","Kochi","Business","India")
]

columns = [
"passenger_id",
"passenger_name",
"city",
"travel_class",
"country"
]

df_day2 = spark.createDataFrame(
    passengers_day2,
    columns
)

display(df_day2)

passenger_id,passenger_name,city,travel_class,country
102,Priya Reddy,Bangalore,First Class,India
104,Sneha Patel,Hyderabad,Premium Economy,India
106,Neha Singh,Pune,Economy,India
107,Arjun Verma,Kochi,Business,India


In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(
    spark,
    "/tmp/passengers_delta"
)

delta_table.alias("target") \
.merge(
    df_day2.alias("source"),
    "target.passenger_id = source.passenger_id"
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
delta_table.alias("target") \
.merge(
    df_day2.alias("source"),
    "target.passenger_id = source.passenger_id"
) \
.whenMatchedUpdateAll() \
.execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
delta_table.alias("target") \
.merge(
    df_day2.alias("source"),
    "target.passenger_id = source.passenger_id"
) \
.whenNotMatchedInsertAll() \
.execute()

In [0]:
spark.read.format("delta") \
    .load("/tmp/passengers_delta") \
    .filter("passenger_id = 102") \
    .show()

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore| First Class|  India|
+------------+--------------+---------+------------+-------+



In [0]:
spark.read.format("delta") \
    .load("/tmp/passengers_delta") \
    .filter("passenger_id = 106") \
    .show()

+------------+--------------+----+------------+-------+
|passenger_id|passenger_name|city|travel_class|country|
+------------+--------------+----+------------+-------+
|         106|    Neha Singh|Pune|     Economy|  India|
+------------+--------------+----+------------+-------+



In [0]:
df_v0 = spark.read.format("delta") \
    .option("versionAsOf",0) \
    .load("/tmp/passengers_delta")

display(df_v0)

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
102,Priya Reddy,Bangalore,Business,India
103,Amit Kumar,Mumbai,Economy,India
104,Sneha Patel,Delhi,Premium Economy,India
105,Farhan Ali,Chennai,Economy,India


In [0]:

df_latest = spark.read.format("delta") \
    .load("/tmp/passengers_delta")

display(df_latest)

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
103,Amit Kumar,Mumbai,Economy,India
105,Farhan Ali,Chennai,Economy,India
102,Priya Reddy,Bangalore,First Class,India
104,Sneha Patel,Hyderabad,Premium Economy,India
106,Neha Singh,Pune,Economy,India
107,Arjun Verma,Kochi,Business,India


In [0]:
print("Version 0")
display(df_v0)

print("Latest Version")
display(df_latest)

Version 0


passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
102,Priya Reddy,Bangalore,Business,India
103,Amit Kumar,Mumbai,Economy,India
104,Sneha Patel,Delhi,Premium Economy,India
105,Farhan Ali,Chennai,Economy,India


Latest Version


passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
103,Amit Kumar,Mumbai,Economy,India
105,Farhan Ali,Chennai,Economy,India
102,Priya Reddy,Bangalore,First Class,India
104,Sneha Patel,Hyderabad,Premium Economy,India
106,Neha Singh,Pune,Economy,India
107,Arjun Verma,Kochi,Business,India


In [0]:
print("Version 0")

spark.read.format("delta") \
.option("versionAsOf",0) \
.load("/tmp/passengers_delta") \
.filter("passenger_id = 102") \
.show()

print("Latest")

spark.read.format("delta") \
.load("/tmp/passengers_delta") \
.filter("passenger_id = 102") \
.show()

Version 0
+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore|    Business|  India|
+------------+--------------+---------+------------+-------+

Latest
+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore| First Class|  India|
+------------+--------------+---------+------------+-------+



In [0]:
print("Version 0")

spark.read.format("delta") \
.option("versionAsOf",0) \
.load("/tmp/passengers_delta") \
.filter("passenger_id = 104") \
.show()

print("Latest")

spark.read.format("delta") \
.load("/tmp/passengers_delta") \
.filter("passenger_id = 104") \
.show()

Version 0
+------------+--------------+-----+---------------+-------+
|passenger_id|passenger_name| city|   travel_class|country|
+------------+--------------+-----+---------------+-------+
|         104|   Sneha Patel|Delhi|Premium Economy|  India|
+------------+--------------+-----+---------------+-------+

Latest
+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
%sql OPTIMIZE delta.`/tmp/passengers_delta`;

path,metrics
dbfs:/tmp/passengers_delta,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1781703086662, 1781703087309, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null)"


In [0]:
%sql OPTIMIZE delta.`/tmp/passengers_delta`
ZORDER BY (city);

path,metrics
dbfs:/tmp/passengers_delta,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 1879), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1781703105090, 1781703105788, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null)"


In [0]:
%sql DELETE FROM delta.`/tmp/passengers_delta`
WHERE passenger_id = 105;

num_affected_rows
1


In [0]:
delta_table.history().show(truncate=False)

+-------+-------------------+---------------+-------------------------------------------------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
%sql VACUUM delta.`/tmp/passengers_delta`;

path
dbfs:/tmp/passengers_delta
